# JSN 评估与最佳狭窄阈值分析

1. 加载带标签结果，与算法预测对比（多标签分类指标）
2. **四个区室分开**：用 label 对应的 JSN (mm) 找每个区室的最优 `jsn_narrow_mm`
3. **内侧 / 外侧分开**：用 label 对应的 JSN 找内侧、外侧各自的最优 `jsn_narrow_mm`

## 配置与导入

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

_nb = Path.cwd().resolve()
KNEE_PKG_ROOT = _nb.parent if _nb.name == "notebooks" else _nb
if str(KNEE_PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(KNEE_PKG_ROOT))

JSN_LABEL_DIR = Path("/path/to/your/jsn_eval_labels")
print("JSN label dir:", JSN_LABEL_DIR)

## 加载带标签结果与多标签评估

In [ ]:
# 加载带标签结果（优先 CSV，否则 Excel）
csv_path = JSN_LABEL_DIR / "jwd_result_w_label.csv"
xlsx_path = JSN_LABEL_DIR / "jsn_results_w_labels.xlsx"
if csv_path.exists():
    df_eval = pd.read_csv(csv_path, encoding="utf-8")
    print(f"Loaded {len(df_eval)} rows from {csv_path}")
elif xlsx_path.exists():
    df_eval = pd.read_excel(xlsx_path)
    print(f"Loaded {len(df_eval)} rows from {xlsx_path}")
else:
    raise FileNotFoundError(f"No label file found in {JSN_LABEL_DIR}")

# 忽略的 case_id（不参与评估）
IGNORE_CASE_IDS = {
    "KOA_卓大均10880012_074Y_F",
    "KOA_周显智10933847_055Y_M",
    "KOA_赵秀兰10689968_070Y_F",
}
case_id_col = "case_id"
if case_id_col in df_eval.columns:
    df_eval = df_eval[~df_eval[case_id_col].isin(IGNORE_CASE_IDS)].copy()
    print(f"Excluded {len(IGNORE_CASE_IDS)} cases; remaining: {len(df_eval)} rows")

# 预测列、标签列、JSN mm 列
compartments = ["left_medial", "left_lateral", "right_medial", "right_lateral"]
pred_cols = [f"{c}_narrow" for c in compartments]
label_cols = [f"{c}_narrow_label" for c in compartments]
mm_cols = [f"{c}_mm" for c in compartments]
for pc, lc in zip(pred_cols, label_cols):
    if pc not in df_eval.columns or lc not in df_eval.columns:
        raise KeyError(f"Missing columns: need {pred_cols} and {label_cols}")

# 标签规则：没写 = false，1 = true
y_pred = df_eval[pred_cols].astype(int)
y_true = df_eval[label_cols].copy()
y_true = (y_true == 1).astype(int)
y_true = y_true.fillna(0).astype(int)
y_true_v = y_true
y_pred_v = y_pred
print(f"Evaluating {len(df_eval)} samples")

# ---------- 多标签测评指标 ----------
def multilabel_metrics(y_true, y_pred, labels):
    from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, hamming_loss
    out = {}
    for i, name in enumerate(labels):
        out[name] = {
            "precision": precision_score(y_true.iloc[:, i], y_pred.iloc[:, i], zero_division=0),
            "recall": recall_score(y_true.iloc[:, i], y_pred.iloc[:, i], zero_division=0),
            "f1": f1_score(y_true.iloc[:, i], y_pred.iloc[:, i], zero_division=0),
            "accuracy": accuracy_score(y_true.iloc[:, i], y_pred.iloc[:, i]),
        }
    out["macro"] = {
        "precision": np.mean([out[n]["precision"] for n in labels]),
        "recall": np.mean([out[n]["recall"] for n in labels]),
        "f1": np.mean([out[n]["f1"] for n in labels]),
        "accuracy": np.mean([out[n]["accuracy"] for n in labels]),
    }
    y_t = y_true.values.ravel()
    y_p = y_pred.values.ravel()
    out["micro"] = {
        "precision": precision_score(y_t, y_p, zero_division=0),
        "recall": recall_score(y_t, y_p, zero_division=0),
        "f1": f1_score(y_t, y_p, zero_division=0),
        "accuracy": accuracy_score(y_t, y_p),
    }
    out["hamming_loss"] = hamming_loss(y_true, y_pred)
    out["subset_accuracy"] = (np.asarray(y_true) == np.asarray(y_pred)).all(axis=1).mean()
    return out

metrics = multilabel_metrics(y_true_v, y_pred_v, compartments)
print("\n=== Per-compartment ===")
for name in compartments:
    m = metrics[name]
    print(f"  {name}: P={m['precision']:.4f}, R={m['recall']:.4f}, F1={m['f1']:.4f}, Acc={m['accuracy']:.4f}")
print("\n=== Macro / Micro ===")
print(f"  Macro: P={metrics['macro']['precision']:.4f}, R={metrics['macro']['recall']:.4f}, F1={metrics['macro']['f1']:.4f}, Acc={metrics['macro']['accuracy']:.4f}")
print(f"  Micro: P={metrics['micro']['precision']:.4f}, R={metrics['micro']['recall']:.4f}, F1={metrics['micro']['f1']:.4f}, Acc={metrics['micro']['accuracy']:.4f}")
print(f"  Subset accuracy: {metrics['subset_accuracy']:.4f}")

df_metrics = pd.DataFrame([{**{"compartment": c}, **metrics[c]} for c in compartments])
df_metrics = pd.concat([df_metrics,
    pd.DataFrame([{"compartment": "macro", **metrics["macro"]}]),
    pd.DataFrame([{"compartment": "micro", **metrics["micro"]}]),
], ignore_index=True)
display(df_metrics)

JSN_LABEL_DIR.mkdir(parents=True, exist_ok=True)
df_metrics.to_csv(JSN_LABEL_DIR / "jsn_evaluation_metrics.csv", index=False, encoding="utf-8")
print(f"\nSaved to {JSN_LABEL_DIR / 'jsn_evaluation_metrics.csv'}")

## 1. 四个区室分开：What is the best jsn_narrow_mm?

用 **label** 作为金标准，对每个区室的 **JSN (mm)** 做二分类：预测规则为 `narrow = (jsn_mm < threshold)`。在阈值网格上搜索使 F1 最大的阈值，作为该区室的最优 `jsn_narrow_mm`。

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

def find_best_threshold(jsn_mm, y_label, thresholds=None):
    """jsn_mm, y_label: 1d array. 预测 narrow = (jsn_mm < t). 返回 (best_t, best_f1, metrics_at_best)."""
    jsn_mm = np.asarray(jsn_mm, dtype=float)
    y_label = np.asarray(y_label, dtype=int)
    valid = ~np.isnan(jsn_mm)
    jsn_mm = jsn_mm[valid]
    y_label = y_label[valid]
    if len(jsn_mm) == 0:
        return np.nan, 0.0, {}
    if thresholds is None:
        th = np.unique(np.percentile(jsn_mm, np.linspace(0, 100, 101)))
        th = np.sort(np.r_[th, np.linspace(jsn_mm.min(), jsn_mm.max(), 200)])
        th = np.unique(th)
    else:
        th = np.asarray(thresholds)
    best_f1 = -1
    best_t = np.nan
    for t in th:
        y_pred_t = (jsn_mm < t).astype(int)
        f1 = f1_score(y_label, y_pred_t, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    y_pred_best = (jsn_mm < best_t).astype(int)
    metrics_at_best = {
        "precision": precision_score(y_label, y_pred_best, zero_division=0),
        "recall": recall_score(y_label, y_pred_best, zero_division=0),
        "f1": best_f1,
        "accuracy": accuracy_score(y_label, y_pred_best),
    }
    return float(best_t), float(best_f1), metrics_at_best

print("=== Best jsn_narrow_mm per compartment (using label vs JSN mm) ===")
best_per_compartment = {}
for c in compartments:
    mm_col = f"{c}_mm"
    label_col = f"{c}_narrow_label"
    if mm_col not in df_eval.columns or label_col not in df_eval.columns:
        print(f"  {c}: missing {mm_col} or {label_col}")
        continue
    y_lab = (df_eval[label_col] == 1).fillna(0).astype(int)
    best_t, best_f1, m = find_best_threshold(df_eval[mm_col].values, y_lab.values)
    best_per_compartment[c] = {"best_jsn_narrow_mm": best_t, "f1": best_f1, **m}
    print(f"  {c}: best_jsn_narrow_mm = {best_t:.3f} mm  (F1={best_f1:.4f}, P={m['precision']:.4f}, R={m['recall']:.4f}, Acc={m['accuracy']:.4f})")

df_best_4 = pd.DataFrame([{"compartment": c, **best_per_compartment[c]} for c in compartments])
display(df_best_4)

## 2. 内侧 vs 外侧分开：What is the best jsn_narrow_mm?

将 **内侧**（left_medial + right_medial）的 (JSN mm, label) 合并为一组，**外侧**（left_lateral + right_lateral）合并为另一组。分别在两组上搜索使 F1 最大的阈值，作为内侧/外侧各自的最优 `jsn_narrow_mm`。

In [ ]:
def pool_jsn_and_label(df, mm_keys, label_keys):
    """Pool (jsn_mm, label) from multiple columns. mm_keys/label_keys: list of column names."""
    jsn_list, lab_list = [], []
    for mk, lk in zip(mm_keys, label_keys):
        if mk not in df.columns or lk not in df.columns:
            continue
        jsn_list.append(df[mk].values)
        lab_list.append((df[lk] == 1).fillna(0).astype(int).values)
    jsn = np.concatenate(jsn_list)
    lab = np.concatenate(lab_list)
    return jsn, lab

medial_mm = ["left_medial_mm", "right_medial_mm"]
medial_label = ["left_medial_narrow_label", "right_medial_narrow_label"]
lateral_mm = ["left_lateral_mm", "right_lateral_mm"]
lateral_label = ["left_lateral_narrow_label", "right_lateral_narrow_label"]

jsn_med, lab_med = pool_jsn_and_label(df_eval, medial_mm, medial_label)
jsn_lat, lab_lat = pool_jsn_and_label(df_eval, lateral_mm, lateral_label)

print("=== Best jsn_narrow_mm: Medial (left_medial + right_medial) ===")
best_t_med, best_f1_med, m_med = find_best_threshold(jsn_med, lab_med)
print(f"  best_jsn_narrow_mm = {best_t_med:.3f} mm  (F1={best_f1_med:.4f}, P={m_med['precision']:.4f}, R={m_med['recall']:.4f}, Acc={m_med['accuracy']:.4f})")

print("\n=== Best jsn_narrow_mm: Lateral (left_lateral + right_lateral) ===")
best_t_lat, best_f1_lat, m_lat = find_best_threshold(jsn_lat, lab_lat)
print(f"  best_jsn_narrow_mm = {best_t_lat:.3f} mm  (F1={best_f1_lat:.4f}, P={m_lat['precision']:.4f}, R={m_lat['recall']:.4f}, Acc={m_lat['accuracy']:.4f})")

df_best_med_lat = pd.DataFrame([
    {"group": "medial", "best_jsn_narrow_mm": best_t_med, "f1": best_f1_med, **m_med},
    {"group": "lateral", "best_jsn_narrow_mm": best_t_lat, "f1": best_f1_lat, **m_lat},
])
display(df_best_med_lat)

# 可选：保存最佳阈值结果
JSN_LABEL_DIR.mkdir(parents=True, exist_ok=True)
df_best_4.to_csv(JSN_LABEL_DIR / "jsn_best_threshold_per_compartment.csv", index=False, encoding="utf-8")
df_best_med_lat.to_csv(JSN_LABEL_DIR / "jsn_best_threshold_medial_lateral.csv", index=False, encoding="utf-8")
print(f"\nSaved best thresholds to {JSN_LABEL_DIR}")